# Combined Output — SOC + ET + CI

In [ ]:
import pathlib
import sys
import os

import anthropic
import pandas as pd
from IPython.display import display, Markdown

_here = pathlib.Path().resolve()
if (_here / 'clinical-trial-optimizer').exists():
    CT_ROOT = _here / 'clinical-trial-optimizer'
elif (_here / 'src').exists():
    CT_ROOT = _here
elif (_here.parent / 'src').exists():
    CT_ROOT = _here.parent
else:
    raise RuntimeError(f'Cannot locate clinical-trial-optimizer root from {_here}')

sys.path.insert(0, str(CT_ROOT / 'src'))

from agent_prompt_functions import get_soc_agent_prompt, get_et_agent_prompt, get_ci_agent_prompt
from data_utils import filterSOCLOT, formatSOCLotDF, filterETLOT, formatETLotDF
from ct_api import fetch_competing_trials

print(f'CT_ROOT : {CT_ROOT}')

In [ ]:
print('API key set' if os.environ.get('ANTHROPIC_API_KEY') else 'WARNING: ANTHROPIC_API_KEY not set')

In [ ]:
# Paste I/E criteria here
ie_criteria = ""


In [ ]:
DATA_FILE = CT_ROOT / 'data' / 'rwd_lot.xlsx'
PERIOD1_RANGE = '2017-2020'
PERIOD2_RANGE = '2021-2024'

sheets = pd.read_excel(DATA_FILE, sheet_name=None)
print(f'Sheets found: {list(sheets.keys())}')

# SOC prompt
_, topN_soc = filterSOCLOT(sheets['soc'], pct=0.03, N=5)
soc_df = formatSOCLotDF(topN_soc)
soc_prompt = get_soc_agent_prompt(ie_criteria, soc_df)

# ET prompt
et_raw = filterETLOT(sheets['period1'], sheets['period2'])
et_df = formatETLotDF(et_raw)
et_prompt = get_et_agent_prompt(ie_criteria, et_df, period1_range=PERIOD1_RANGE, period2_range=PERIOD2_RANGE)

# CI prompt — ⚠️ HUMAN APPROVAL REQUIRED: makes live API call to ClinicalTrials.gov
competing_trials_md = fetch_competing_trials(ie_criteria)
ci_prompt = get_ci_agent_prompt(ie_criteria, competing_trials_md)

print(f'SOC prompt : {len(soc_prompt):,} chars')
print(f'ET  prompt : {len(et_prompt):,} chars')
print(f'CI  prompt : {len(ci_prompt):,} chars')

In [ ]:
# ⚠️ HUMAN APPROVAL REQUIRED — this cell calls the Claude API (3 requests)
client = anthropic.Anthropic()

for label, prompt in [
    ('## SOC Agent — Standard of Care Analysis', soc_prompt),
    ('## ET Agent — Evolving Treatment Analysis', et_prompt),
    ('## CI Agent — Competitive Intelligence Analysis', ci_prompt),
]:
    display(Markdown(label))
    response = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=4096,
        messages=[{'role': 'user', 'content': prompt}],
    )
    display(Markdown(response.content[0].text))